# Chapter 12 -- Head-to-Head Benchmark

Companion notebook for `affinity/book/chapters/CH12_benchmark.tex`.

Loads `affinity/book/data/cached_audits/benchmark_360cell.json` (ported via
`tabkernels.audits.benchmark.migrate_flagship_json` from the flagship sweep)
and reproduces:

* Table 12.1 -- per-corner winner counts
* Table 12.2 -- NW vs MLP head-mode comparison
* Figure 12.1 -- per-cell win-rate scatter (saved to `affinity/book/figures/`)

It also runs a small live demo of `run_360cell_benchmark` (3 datasets x 3
corners x 1 seed, `slice_only=True`) to demonstrate the public API. The full
360-cell sweep is multi-GPU-hour and is documented in the markdown cell at
the bottom on how to scale up via TALENT.

Notebook runtime under papermill: < 30 minutes (in practice << 1 minute,
since the heavy lifting is amortized by the cached JSON).

In [ ]:
# Cell 1: setup
import json
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tabkernels.audits import (
    Corner,
    aggregate_winrates,
    plot_per_corner_winners,
    run_360cell_benchmark,
)
from tabkernels.audits.benchmark import (
    ALL_VARIANTS,
    ASYMMETRIC_VARIANTS,
    SYMMETRIC_VARIANTS,
    _headtohead_per_cell,
)

BOOK = Path('/home/asudjianto/jupyterlab/similarity-hierarchy-research/affinity/book')
CACHED = BOOK / 'data' / 'cached_audits' / 'benchmark_360cell.json'
FIG_OUT = BOOK / 'figures' / 'fig_12_01_per_cell_winrate.pdf'

report = json.loads(CACHED.read_text())
print('Loaded', len(report['per_seed']), 'cells from', CACHED.name)
print('Headline overall sym_win_frac:',
      f"{report['metrics']['headline_overall_sym_win_frac']:.3f}")
for hm, payload in report['metrics']['by_head_mode'].items():
    print(f"  {hm}: n={int(payload['n'])} sym_win_frac={payload['sym_win_frac']:.3f}")

## Cell 2 -- Aggregate the head-to-head DataFrame

`aggregate_winrates(report)` returns one row per (head_mode, depth, task)
corner, plus three roll-up rows (NW overall, MLP overall, all corners). The
headline 78% / 88% / 69% rates appear as the three roll-up rows.

In [ ]:
# Cell 2: per-corner head-to-head table.
df = aggregate_winrates(report)
with pd.option_context('display.precision', 3):
    print(df.to_string(index=False))

## Cell 3 -- Table 12.1: per-corner winner counts

The flagship table.  Each row is a (head_mode, depth, task) corner; each
column is a variant; entries are the count of cells (out of 30 per corner)
where that variant has the lowest mean held-out loss.

In [ ]:
# Cell 3: Table 12.1.
pcc = report['metrics']['per_corner_winner_counts']
order = list(ASYMMETRIC_VARIANTS) + list(SYMMETRIC_VARIANTS)
rows = []
for hm in ('nw', 'mlp'):
    for L in (1, 2, 4):
        for task in ('regression', 'classification'):
            counts = pcc[f'{hm}|{L}|{task}']['counts']
            rows.append(dict(
                HM=hm, L=L, Task=task[:3],
                **{v: counts.get(v, 0) for v in order},
            ))
tbl12_1 = pd.DataFrame(rows)
print('Table 12.1 -- Per-corner winner counts:')
print(tbl12_1.to_string(index=False))

# Variant total wins across the 360-cell grid.
totals = {v: 0 for v in order}
for _, payload in pcc.items():
    for v, c in payload['counts'].items():
        totals[v] = totals.get(v, 0) + c
print()
print('Total wins (out of 360 cells):')
for v, c in sorted(totals.items(), key=lambda x: -x[1]):
    print(f'  {v:18s} {c:4d}  ({c / 360:.0%})')

In [ ]:
# Cell 4: Table 12.1b -- per-corner winrate (Pr[Delta > 0], median Delta).
h2h = _headtohead_per_cell(report['per_seed'])
rows = []
for hm in ('nw', 'mlp'):
    for L in (1, 2, 4):
        for task in ('regression', 'classification'):
            sub = [r for r in h2h
                   if r['head_mode'] == hm
                   and r['n_layers'] == L
                   and r['task'] == task]
            wins = sum(int(r['sym_wins']) for r in sub)
            rows.append(dict(
                HM=hm, L=L, Task=task[:3],
                pr_sym_beats=wins / len(sub) if sub else float('nan'),
                median_delta=float(np.median([r['delta'] for r in sub])) if sub else float('nan'),
                n=len(sub),
            ))
tbl12_1b = pd.DataFrame(rows)
print('Per-corner: Pr[Delta > 0] and median Delta')
with pd.option_context('display.precision', 4):
    print(tbl12_1b.to_string(index=False))

## Cell 5 -- Figure 12.1: per-cell win-rate scatter

Each marker is one (dataset, $H$, $L$, head-mode) cell, plotted at
($x = L_\text{Std-Attn}$, $y = \min_v L_v$). Markers below the diagonal
are cells where the best symmetric variant beats Std-Attn. The figure is
saved to `affinity/book/figures/fig_12_01_per_cell_winrate.pdf` for the
chapter PDF.

In [ ]:
# Cell 5: Figure 12.1.
fig = plot_per_corner_winners(report)
FIG_OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_OUT, bbox_inches='tight')
print('Saved Figure 12.1 to', FIG_OUT)

## Cell 6 -- Table 12.2: NW vs MLP head-mode comparison

Median paired delta $L_\text{MLP} - L_\text{NW}$ per variant, aggregated over
the 90 (dataset, $H$, $L$) cells per task type. Std-Attn is essentially
indifferent to the head swap ($|\Delta L| \le 0.005$); PSD-NW improves on
89% of paired cells under MLP.

In [ ]:
# Cell 6: Table 12.2.
from collections import defaultdict

cells = report['per_seed']
rows = []
for variant in (list(ASYMMETRIC_VARIANTS) + list(SYMMETRIC_VARIANTS)):
    paired = defaultdict(dict)
    for r in cells:
        if r['variant'] != variant:
            continue
        key = (r['dataset'], r['H'], r['n_layers'])
        paired[key][r['head_mode']] = (r['mean'], r['task'])
    deltas_reg, deltas_cla = [], []
    mlp_wins = total = 0
    for _, by_mode in paired.items():
        if 'nw' not in by_mode or 'mlp' not in by_mode:
            continue
        nw_loss, task = by_mode['nw']
        mlp_loss, _ = by_mode['mlp']
        delta = mlp_loss - nw_loss
        if task == 'regression':
            deltas_reg.append(delta)
        else:
            deltas_cla.append(delta)
        total += 1
        mlp_wins += int(mlp_loss < nw_loss)
    rows.append(dict(
        variant=variant,
        median_dL_reg=float(np.median(deltas_reg)),
        median_dL_cla=float(np.median(deltas_cla)),
        pct_mlp_wins=mlp_wins / total if total else float('nan'),
    ))
tbl12_2 = pd.DataFrame(rows)
print('Table 12.2 -- Paired NW vs MLP per variant:')
with pd.option_context('display.precision', 4):
    print(tbl12_2.to_string(index=False))

In [ ]:
# Cell 7: median NW loss by depth (sensitivity to depth, supports Table 12.3).
rows = []
for variant in (list(ASYMMETRIC_VARIANTS) + list(SYMMETRIC_VARIANTS)):
    row = dict(variant=variant)
    for task, suffix in (('regression', 'reg'), ('classification', 'cla')):
        for L in (1, 2, 4):
            losses = [c['mean'] for c in cells
                      if c['variant'] == variant
                      and c['head_mode'] == 'nw'
                      and c['task'] == task
                      and c['n_layers'] == L]
            row[f'{suffix}_L{L}'] = float(np.median(losses))
    rows.append(row)
tbl_depth = pd.DataFrame(rows)
print('Median NW held-out loss by depth (per variant):')
with pd.option_context('display.precision', 3):
    print(tbl_depth.to_string(index=False))

In [ ]:
# Cell 8: best-symmetric variant identity per (head_mode, task) stratum.
rows = []
for hm in ('nw', 'mlp'):
    for task in ('regression', 'classification'):
        sub = [r for r in h2h if r['head_mode'] == hm and r['task'] == task]
        n = len(sub)
        c = Counter(r['best_sym_variant'] for r in sub)
        rows.append(dict(
            HM=hm, Task=task[:3], n=n,
            **{v: c.get(v, 0) / n for v in SYMMETRIC_VARIANTS},
        ))
tbl_sym_best = pd.DataFrame(rows)
print('Best-symmetric variant identity per (head_mode, task) stratum:')
with pd.option_context('display.precision', 2):
    print(tbl_sym_best.to_string(index=False))

## Cell 9 -- Live demo of `run_360cell_benchmark`

The full sweep is multi-GPU-hour and runs the architecture stack of
`tabkernels.architectures` end-to-end on each of the 20 datasets. For a
tractable in-notebook demo, we exercise the same API in `slice_only=True`
mode: the cell-loop runs against deterministic stubs but goes through the
full report assembly + `aggregate_winrates` aggregation pipeline.

In [ ]:
# Cell 9: live demo, 3 datasets x 3 corners x 1 seed.
demo_datasets = [
    dict(name='Iris', task='classification', n=150, p=4, metric='ce'),
    dict(name='Wine', task='classification', n=178, p=13, metric='ce'),
    dict(name='Diabetes', task='regression', n=442, p=10, metric='mse'),
]
demo_corners = [
    Corner(head_mode='nw', depth=1, task='classification'),
    Corner(head_mode='mlp', depth=1, task='classification'),
    Corner(head_mode='mlp', depth=2, task='regression'),
]
demo_report = run_360cell_benchmark(
    datasets=demo_datasets,
    variants=list(ALL_VARIANTS),
    corners=demo_corners,
    seeds=1,
    head_counts=(1, 4),
    slice_only=True,
)
print('Demo report:', demo_report['config'])
demo_df = aggregate_winrates(demo_report)
print()
print('Demo aggregate:')
with pd.option_context('display.precision', 3):
    print(demo_df.to_string(index=False))

## Scaling up via TALENT

The full 360-cell sweep against real architectures is multi-GPU-hour even on
an NVIDIA GB10. To scale beyond the 20-dataset suite of this chapter, the
recommended path is to wire `run_360cell_benchmark` into the
[TALENT](https://github.com/qile2000/LAMDA-TALENT) tabular benchmark harness:

```python
from talent.benchmarks import load_talent_datasets
from tabkernels.audits import Corner, run_360cell_benchmark
from my_train import train_attention_block  # provides the train_fn signature

datasets = load_talent_datasets(suite='small_tabular', n_max=1500)
corners = [Corner(head_mode=hm, depth=L, task=t)
           for hm in ('nw', 'mlp') for L in (1, 2, 4)
           for t in ('regression', 'classification')]
report = run_360cell_benchmark(
    datasets=datasets,
    variants=ALL_VARIANTS,
    corners=corners,
    seeds=3,
    slice_only=False,
    train_fn=train_attention_block,
    cache_path='cached_audits/benchmark_talent.json',
)
```

The `train_fn` takes
`(dataset, variant, head_mode, n_layers, H, seed, epochs)` and returns a
dict with at least `loss` and `mean_acc`. See the `affinity/ablation_runner.py`
script in the flagship paper repository for the reference implementation
this notebook's cached JSON was produced from.

*(Notebook end. The chapter PDF reads everything from the cached JSON;
this notebook is the regeneration recipe.)*